In [ ]:
!pip install transformers trl accelerate bitsandbytes peft -q

In [ ]:
import torch
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOTrainer, PPOConfig

base_model = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    base_model,
    torch_dtype=torch.float32,
    device_map="auto"
)

ppo_config = PPOConfig(
    batch_size=2,
    mini_batch_size=1,
    learning_rate=1e-5,
)

prompts = ["Say the magic word:", "Output something nice:"]
dataset = [{"prompt": p} for p in prompts]

trainer = PPOTrainer(
    ppo_config,
    model,
    tokenizer,
    dataset,
)

def toy_reward_fn(texts):
    return [1.0 if "success" in t.lower() else 0.0 for t in texts]

# train for a single batch (2 examples)
for batch in trainer.dataloader:
    queries = batch["prompt"]
    query_tensors = tokenizer(
        queries, return_tensors="pt", padding=True
    ).input_ids.to(model.device)

    response_tensors = trainer.generate(
        query_tensors,
        max_new_tokens=20,
        do_sample=True,
        top_p=0.9,
    )

    responses = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)
    rewards = toy_reward_fn(responses)

    # PPO update
    trainer.step(query_tensors, response_tensors, torch.tensor(rewards).to(model.device))
